In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np
from transformers import AutoTokenizer
from torch_geometric.nn import GCNConv
from torch_geometric.utils import add_self_loops
import random
import evaluate  # <--- Added for metrics

# ... (Keep all your model class definitions: SpatioTemporalEEGEncoder, Decoder, etc. unchanged) ...

# ==================================================================================
# MAIN EXECUTION
# ==================================================================================

if __name__ == "__main__":
    # --- CONFIGURATION ---
    NUM_SAMPLES = 10  # <--- Change this to test more/fewer samples
    H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
    MODEL_WEIGHTS_PATH = "eeg-text-max-bleu-v2.pt"
    LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
    
    # Setup
    device = torch.device("cpu")
    print(f"Running Inference on: {device}")
    
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
    PAD_ID = tokenizer.pad_token_id
    SOS_ID = tokenizer.cls_token_id
    EOS_ID = tokenizer.sep_token_id
    TEXT_VOCAB_SIZE = tokenizer.vocab_size
    NUM_COLORS = 9       
    NUM_OBJECTS = 6 

    # 1. Initialize Model
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE, num_colors=NUM_COLORS, num_objects=NUM_OBJECTS,
        pad_id=PAD_ID, dropout=0.0, enc_hidden=256, dec_hidden=256, emb_dim=256
    ).to(device)

    # 2. Load Weights
    try:
        checkpoint = torch.load(MODEL_WEIGHTS_PATH, map_location=device)
        model.load_state_dict(checkpoint)
        print(f"Loaded weights from {MODEL_WEIGHTS_PATH}")
    except FileNotFoundError:
        print(f"Error: {MODEL_WEIGHTS_PATH} not found.")
        exit()

    # 3. Create Static Graph
    edge_index = torch.combinations(torch.arange(62), r=2).t().contiguous()
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    edge_index, _ = add_self_loops(edge_index, num_nodes=62)
    edge_index = edge_index.to(device)
    edge_attr = torch.ones(edge_index.shape[1]).to(device)

    # 4. Load Data & Run Inference
    print(f"Opening dataset: {H5_FILE_PATH}")
    
    # Load Metrics
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    
    predictions = []
    references = []

    with h5py.File(H5_FILE_PATH, 'r') as f:
        total_samples = f['eeg'].shape[0]
        
        # Select the last N samples (or use random.sample for random selection)
        if NUM_SAMPLES > total_samples:
            print(f"Requested {NUM_SAMPLES} samples but dataset only has {total_samples}.")
            NUM_SAMPLES = total_samples
            
        test_indices = random.sample(range(total_samples), NUM_SAMPLES)
        
        print(f"\n--- Running Inference on {NUM_SAMPLES} samples (Oracle Metadata) ---")
        
        for idx in test_indices:
            # Load raw data
            eeg_data = torch.from_numpy(f['eeg'][idx].astype(np.float32)).to(device)
            meta_data = torch.from_numpy(f['metadata'][idx].astype(np.float32)).to(device)
            text_data = f['input_ids'][idx].astype(np.int64)
            
            # Decode Ground Truth Text
            gt_text = tokenizer.decode(text_data, skip_special_tokens=True)
            
            # Run Inference (Oracle)
            pred_text = beam_search_decoder_cpu(model, eeg_data, meta_data, edge_index, edge_attr)
            
            # Store for metrics
            predictions.append(pred_text)
            references.append(gt_text)
            
            print(f"Sample {idx}:")
            print(f"  GT:   {gt_text}")
            print(f"  Pred: {pred_text}")
            print("-" * 30)

    # 5. Compute Scores
    print("\n--- Evaluation Metrics ---")
    
    # BLEU expects references as a list of lists [[ref1], [ref2]]
    bleu_score = bleu.compute(predictions=predictions, references=[[r] for r in references])
    
    # ROUGE expects references as a list of strings [ref1, ref2]
    rouge_score = rouge.compute(predictions=predictions, references=references)
    
    print(f"BLEU Score:    {bleu_score['bleu']:.4f}")
    print(f"ROUGE-1:       {rouge_score['rouge1']:.4f}")
    print(f"ROUGE-2:       {rouge_score['rouge2']:.4f}")
    print(f"ROUGE-L:       {rouge_score['rougeL']:.4f}")

Running Inference on: cpu
Loaded weights from eeg-text-max-bleu-v2.pt
Opening dataset: /home/poorna/data/eeg_dataset_1400_multilabel.h5

--- Running Inference on 10 samples (Oracle Metadata) ---
Sample 27062:
  GT:   a forest with lots of trees and leaves
  Pred: a forest with lots of trees and trees
------------------------------
Sample 26182:
  GT:   a boat is moving through the water
  Pred: a large car is driving the a
------------------------------
Sample 18207:
  GT:   a beach with palm trees and a body of water
  Pred: a city with a a with and trees
------------------------------
Sample 3917:
  GT:   a sport car drives on the highway
  Pred: a large car is driving the a
------------------------------
Sample 3185:
  GT:   a person riding a motorcycle
  Pred: a person riding a motorcycle down a street
------------------------------
Sample 1609:
  GT:   a man in a black shirt playing drums in a room
  Pred: a woman in a boxing ring
------------------------------
Sample 1689:
  GT: 